In [3]:
# %%
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.utils import make_grid, save_image
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from pathlib import Path
import math
from tqdm import tqdm
import random

# %%
# Configurações do dispositivo
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Hiperparâmetros
BATCH_SIZE = 32
IMAGE_SIZE = 64
NC = 1  # Número de canais (1 para grayscale)
NUM_EPOCHS = 100
LR = 1e-4
TIMESTEPS = 100  # Otimização: menos passos, mais rápido
BETA_START = 0.0001
BETA_END = 0.02

# %%
class MedicalImageDataset(Dataset):
    """Dataset customizado para imagens médicas com cache opcional em RAM"""
    def __init__(self, image_paths, transform=None, cache_in_ram=False):
        self.image_paths = image_paths
        self.transform = transform
        self.cache_in_ram = cache_in_ram
        self.cached_images = []
        if self.cache_in_ram:
            print("Carregando todas as imagens na RAM... (pode demorar na primeira vez)")
            for path in tqdm(self.image_paths, desc="Cachando imagens"):
                img = Image.open(path).convert('L')
                self.cached_images.append(img)
            print(f"{len(self.cached_images)} imagens carregadas em RAM.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        if self.cache_in_ram:
            image = self.cached_images[idx]
        else:
            image_path = self.image_paths[idx]
            image = Image.open(image_path).convert('L')
        if self.transform:
            image = self.transform(image)
        return image

# %%
class DiffusionScheduler:
    """Scheduler para o processo de difusão"""
    def __init__(self, timesteps=100, beta_start=0.0001, beta_end=0.02):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.sqrt_recipm1_alphas_cumprod = torch.sqrt(1.0 / self.alphas_cumprod - 1)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod[t].reshape(-1, 1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].reshape(-1, 1, 1, 1)
        return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise

# %%
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.gn1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.gn2 = nn.GroupNorm(8, out_channels)
        if time_emb_dim is not None:
            self.time_mlp = nn.Linear(time_emb_dim, out_channels)
        else:
            self.time_mlp = None
        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.residual_conv = nn.Identity()

    def forward(self, x, time_emb=None):
        residual = self.residual_conv(x)
        h = self.conv1(x)
        h = self.gn1(h)
        h = F.relu(h)
        if self.time_mlp is not None and time_emb is not None:
            time_emb = self.time_mlp(time_emb)
            h = h + time_emb[:, :, None, None]
        h = self.conv2(h)
        h = self.gn2(h)
        h = F.relu(h)
        return h + residual

class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.gn = nn.GroupNorm(8, channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.gn(x)
        q = self.q(h)
        k = self.k(h)
        v = self.v(h)
        q = q.reshape(B, C, H*W).permute(0, 2, 1)
        k = k.reshape(B, C, H*W)
        v = v.reshape(B, C, H*W).permute(0, 2, 1)
        attn = torch.bmm(q, k)
        attn = attn * (int(C)**(-0.5))
        attn = F.softmax(attn, dim=2)
        h = torch.bmm(attn, v)
        h = h.permute(0, 2, 1).reshape(B, C, H, W)
        h = self.proj_out(h)
        return x + h

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, time_emb_dim=128):
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.time_mlp = nn.Sequential(
            TimeEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )
        self.conv_in = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.down1 = nn.ModuleList([
            ResidualBlock(64, 64, time_emb_dim),
            ResidualBlock(64, 64, time_emb_dim)
        ])
        self.down1_pool = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.down2 = nn.ModuleList([
            ResidualBlock(128, 128, time_emb_dim),
            ResidualBlock(128, 128, time_emb_dim)
        ])
        self.down2_pool = nn.Conv2d(128, 256, 3, stride=2, padding=1)
        self.down3 = nn.ModuleList([
            ResidualBlock(256, 256, time_emb_dim),
            ResidualBlock(256, 256, time_emb_dim)
        ])
        self.down3_pool = nn.Conv2d(256, 512, 3, stride=2, padding=1)
        self.bottleneck = nn.ModuleList([
            ResidualBlock(512, 512, time_emb_dim),
            AttentionBlock(512),
            ResidualBlock(512, 512, time_emb_dim)
        ])
        self.up3 = nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1)
        self.up3_blocks = nn.ModuleList([
            ResidualBlock(512, 256, time_emb_dim),
            ResidualBlock(256, 256, time_emb_dim)
        ])
        self.up2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.up2_blocks = nn.ModuleList([
            ResidualBlock(256, 128, time_emb_dim),
            ResidualBlock(128, 128, time_emb_dim)
        ])
        self.up1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.up1_blocks = nn.ModuleList([
            ResidualBlock(128, 64, time_emb_dim),
            ResidualBlock(64, 64, time_emb_dim)
        ])
        self.conv_out = nn.Conv2d(64, out_channels, 3, padding=1)

    def forward(self, x, timesteps):
        time_emb = self.time_mlp(timesteps)
        x = self.conv_in(x)
        skip1 = x
        for block in self.down1:
            x = block(x, time_emb)
        x = self.down1_pool(x)
        skip2 = x
        for block in self.down2:
            x = block(x, time_emb)
        x = self.down2_pool(x)
        skip3 = x
        for block in self.down3:
            x = block(x, time_emb)
        x = self.down3_pool(x)
        for block in self.bottleneck:
            if isinstance(block, AttentionBlock):
                x = block(x)
            else:
                x = block(x, time_emb)
        x = self.up3(x)
        x = torch.cat([x, skip3], dim=1)
        for block in self.up3_blocks:
            x = block(x, time_emb)
        x = self.up2(x)
        x = torch.cat([x, skip2], dim=1)
        for block in self.up2_blocks:
            x = block(x, time_emb)
        x = self.up1(x)
        x = torch.cat([x, skip1], dim=1)
        for block in self.up1_blocks:
            x = block(x, time_emb)
        x = self.conv_out(x)
        return x

# %%
class MedicalDiffusionModel:
    def __init__(self, class_name, image_paths, cache_in_ram=False):
        self.class_name = class_name
        self.image_paths = image_paths
        self.transform = transforms.Compose([
            transforms.Resize(IMAGE_SIZE),
            transforms.CenterCrop(IMAGE_SIZE),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        self.dataset = MedicalImageDataset(image_paths, self.transform, cache_in_ram=cache_in_ram)
        self.dataloader = DataLoader(self.dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        self.model = UNet(in_channels=NC, out_channels=NC).to(device)
        self.scheduler = DiffusionScheduler(TIMESTEPS, BETA_START, BETA_END)
        self.optimizer = optim.Adam(self.model.parameters(), lr=LR)
        self.losses = []
        self.scheduler.betas = self.scheduler.betas.to(device)
        self.scheduler.alphas = self.scheduler.alphas.to(device)
        self.scheduler.alphas_cumprod = self.scheduler.alphas_cumprod.to(device)
        self.scheduler.sqrt_alphas_cumprod = self.scheduler.sqrt_alphas_cumprod.to(device)
        self.scheduler.sqrt_one_minus_alphas_cumprod = self.scheduler.sqrt_one_minus_alphas_cumprod.to(device)
        self.scheduler.sqrt_recip_alphas = self.scheduler.sqrt_recip_alphas.to(device)
        self.scheduler.sqrt_recipm1_alphas_cumprod = self.scheduler.sqrt_recipm1_alphas_cumprod.to(device)
        self.scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    def train(self):
        print(f"Iniciando treinamento para classe: {self.class_name}")
        print(f"Número de imagens: {len(self.dataset)}")
        self.model.train()
        for epoch in range(NUM_EPOCHS):
            epoch_loss = 0
            pbar = tqdm(self.dataloader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
            for batch_idx, x_0 in enumerate(pbar):
                x_0 = x_0.to(device, non_blocking=True)
                batch_size = x_0.shape[0]
                t = torch.randint(0, TIMESTEPS, (batch_size,), device=device).long()
                noise = torch.randn_like(x_0)
                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    x_t = self.scheduler.q_sample(x_0, t, noise)
                    predicted_noise = self.model(x_t, t)
                    loss = F.mse_loss(predicted_noise, noise)
                self.optimizer.zero_grad(set_to_none=True)
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                epoch_loss += loss.item()
                self.losses.append(loss.item())
                pbar.set_postfix({'loss': loss.item()})
            avg_loss = epoch_loss / len(self.dataloader)
            print(f'Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {avg_loss:.6f}')
            if epoch % 20 == 0 or epoch == NUM_EPOCHS-1:
                self.save_sample_images(epoch)
                self.save_model(epoch)
        print(f"Treinamento concluído para classe: {self.class_name}")

    @torch.no_grad()
    def sample(self, num_samples=64):
        self.model.eval()
        x = torch.randn(num_samples, NC, IMAGE_SIZE, IMAGE_SIZE).to(device)
        for t in reversed(range(TIMESTEPS)):
            t_tensor = torch.full((num_samples,), t, device=device, dtype=torch.long)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                predicted_noise = self.model(x, t_tensor)
            alpha = self.scheduler.alphas[t]
            alpha_cumprod = self.scheduler.alphas_cumprod[t]
            beta = self.scheduler.betas[t]
            coeff1 = 1 / torch.sqrt(alpha)
            coeff2 = beta / torch.sqrt(1 - alpha_cumprod)
            x = coeff1 * (x - coeff2 * predicted_noise)
            if t > 0:
                noise = torch.randn_like(x)
                x = x + torch.sqrt(beta) * noise
        return x

    def save_sample_images(self, epoch):
        samples = self.sample(num_samples=64)
        samples = (samples + 1) / 2.0
        samples = torch.clamp(samples, 0, 1)
        grid = make_grid(samples, nrow=8, padding=2, normalize=False)
        os.makedirs(f'diffusion_samples_{self.class_name}', exist_ok=True)
        plt.figure(figsize=(10, 10), dpi=150)
        npimg = grid.cpu().numpy()
        plt.imshow(np.transpose(npimg, (1, 2, 0)).squeeze(), cmap='gray')
        plt.axis('off')
        plt.title(f'Generated samples - Epoch {epoch}')
        plt.savefig(f'diffusion_samples_{self.class_name}/samples_epoch_{epoch:03d}.png',
                    bbox_inches='tight', dpi=150)
        plt.close()

    def save_model(self, epoch):
        os.makedirs('models', exist_ok=True)
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_params': {
                'timesteps': TIMESTEPS,
                'beta_start': BETA_START,
                'beta_end': BETA_END
            },
            'epoch': epoch
        }, f'models/diffusion_model_{self.class_name}_epoch{epoch:03d}.pth')

    def generate_images(self, num_images):
        generated_images = []
        for i in range(0, num_images, BATCH_SIZE):
            batch_size = min(BATCH_SIZE, num_images - i)
            samples = self.sample(batch_size)
            samples = (samples + 1) / 2.0
            samples = torch.clamp(samples, 0, 1)
            for j in range(batch_size):
                img_tensor = samples[j]
                img_pil = transforms.ToPILImage()(img_tensor.cpu())
                generated_images.append(img_pil)
        return generated_images

    def plot_losses(self):
        plt.figure(figsize=(10, 5))
        plt.title(f"Perda do Modelo de Difusão - {self.class_name}")
        plt.plot(self.losses)
        plt.xlabel("Iterações")
        plt.ylabel("MSE Loss")
        plt.grid(True)
        plt.savefig(f'diffusion_losses_{self.class_name}.png', dpi=150)
        plt.show()

# %%
def prepare_dataset_paths(base_path):
    dataset_info = {
        'covid19': {'current': 10000, 'target': 15700},
        'normal': {'current': 15700, 'target': 15700},
        'pneumonia_bacterial': {'current': 5500, 'target': 15700},
        'pneumonia_viral': {'current': 3000, 'target': 15700}
    }
    paths = {}
    for class_name in dataset_info.keys():
        class_path = Path(base_path) / class_name
        if class_path.exists():
            paths[class_name] = list(class_path.glob('*.png')) + \
                               list(class_path.glob('*.jpg')) + \
                               list(class_path.glob('*.jpeg'))
        else:
            print(f"Aviso: Diretório {class_path} não encontrado")
            paths[class_name] = []
    return paths, dataset_info

# %%
def balance_dataset(base_path, cache_in_ram=False):
    paths, dataset_info = prepare_dataset_paths(base_path)
    classes_to_augment = ['pneumonia_viral', 'pneumonia_bacterial', 'covid19']
    diffusion_models = {}
    for class_name in classes_to_augment:
        if class_name in paths and len(paths[class_name]) > 0:
            print(f"\n=== Treinando Modelo de Difusão para {class_name} ===")
            model = MedicalDiffusionModel(class_name, paths[class_name], cache_in_ram=cache_in_ram)
            model.train()
            model.plot_losses()
            current_count = dataset_info[class_name]['current']
            target_count = dataset_info[class_name]['target']
            images_to_generate = target_count - current_count
            if images_to_generate > 0:
                print(f"Gerando {images_to_generate} imagens para {class_name}")
                generated_images = model.generate_images(images_to_generate)
                output_dir = Path(f'diffusion_generated_{class_name}')
                output_dir.mkdir(exist_ok=True)
                for i, img in enumerate(generated_images):
                    img.save(output_dir / f'diffusion_generated_{class_name}_{i:04d}.png')
                print(f"Imagens salvas em: {output_dir}")
            diffusion_models[class_name] = model
    return diffusion_models

# %%
def load_trained_model(model_path, class_name):
    model = UNet(in_channels=NC, out_channels=NC).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    scheduler_params = checkpoint['scheduler_params']
    scheduler = DiffusionScheduler(
        scheduler_params['timesteps'],
        scheduler_params['beta_start'],
        scheduler_params['beta_end']
    )
    diffusion_model = MedicalDiffusionModel(class_name, [])
    diffusion_model.model = model
    diffusion_model.scheduler = scheduler
    return diffusion_model

# %%
if __name__ == "__main__":
    BASE_PATH = "./dataSetPibic/train"
    print("Iniciando balanceamento do dataset médico com Modelos de Difusão...")
    print("Dataset atual:")
    print("- COVID-19: 10,000 imagens")
    print("- Normal: 15,700 imagens")
    print("- Pneumonia Bacteriana: 5,500 imagens")
    print("- Pneumonia Viral: 3,000 imagens")
    print("\nObjetivo: 15,700 imagens para cada classe")
    # Ative cache_in_ram=True se quiser usar toda a RAM disponível para acelerar leitura
    trained_models = balance_dataset(BASE_PATH, cache_in_ram=True)

    print("\n=== Vantagens dos Modelos de Difusão sobre GANs ===")
    print("1. Melhor qualidade de imagem")
    print("2. Treinamento mais estável")
    print("3. Maior diversidade nas imagens geradas")
    print("4. Sem problemas de mode collapse")
    print("5. Melhor preservação de detalhes médicos importantes")

    print("\nPara usar este código:")
    print("1. Organize suas imagens na estrutura de pastas especificada")
    print("2. Altere BASE_PATH para o caminho do seu dataset")
    print("3. Execute balance_dataset(BASE_PATH, cache_in_ram=True)")
    print("4. As imagens geradas serão salvas em pastas 'diffusion_generated_[classe]'")
    print("5. Os modelos treinados serão salvos em 'models/' (um checkpoint por época)")
    print("6. Amostras durante treinamento: 'diffusion_samples_[classe]/'")
    print("7. Gráficos de perda: 'diffusion_losses_[classe].png'")

Usando dispositivo: cuda:0
Iniciando balanceamento do dataset médico com Modelos de Difusão...
Dataset atual:
- COVID-19: 10,000 imagens
- Normal: 15,700 imagens
- Pneumonia Bacteriana: 5,500 imagens
- Pneumonia Viral: 3,000 imagens

Objetivo: 15,700 imagens para cada classe

=== Treinando Modelo de Difusão para pneumonia_viral ===
Carregando todas as imagens na RAM... (pode demorar na primeira vez)


Cachando imagens: 100%|██████████| 2978/2978 [00:22<00:00, 129.89it/s]
/tmp/ipykernel_8888/1824588550.py:260: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


2978 imagens carregadas em RAM.
Iniciando treinamento para classe: pneumonia_viral
Número de imagens: 2978


Epoch 1/100:   0%|          | 0/94 [00:00<?, ?it/s]/tmp/ipykernel_8888/1824588550.py:274: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
Epoch 1/100: 100%|██████████| 94/94 [00:09<00:00,  9.63it/s, loss=0.0461]
/tmp/ipykernel_8888/1824588550.py:298: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


Epoch 1/100, Loss: 0.243273


...

Epoch 7/100, Loss: 0.074739


Epoch 8/100: 100%|██████████| 94/94 [00:09<00:00,  9.52it/s, loss=0.0654]


Epoch 8/100, Loss: 0.067992


Epoch 9/100:  11%|█         | 10/94 [00:01<00:11,  7.18it/s, loss=0.0571]


KeyboardInterrupt: 